### Setup
---

In [106]:
import json
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv

In [107]:
client = OpenAI()

In [108]:
load_dotenv(override=True)

True

### Tools
---

In [229]:
source_of_truth = "https://www.formula1.com/en/results/2026/races"

In [236]:
def evaluate_drivers_standings(source_of_truth): 
    print(1)
    validationPrompt = f"""
        You are an F1 data assistant. Retrieve the current F1 driver standings.

        SOURCE OF TRUTH:
        Search and retrieve the current standings from: {source_of_truth}

        Respond with a JSON object in this exact format:
        {{
            "validatedResults": [
            {{
                "position": 1,
                "driver": "Driver name",
                "team": "Team name",
                "points": 0
            }}
        ]
        }}

        Return only the JSON object with no additional text.
    """

    response = client.responses.create(
        model="gpt-5.4-nano",
        tools=[
            {
                "type": "web_search",
                "search_context_size": "medium",
            }
        ],
        input=validationPrompt
    )

    result = json.loads(response.output_text)
    return {"validatedResults": result["validatedResults"]}

In [254]:
tools = [
    {
        "type": "function", 
        "name": "evaluate_drivers_standings",
        "description": """
            Fetches the current official F1 driver championship standings.
            
            Call this tool ONLY when the user asks about:
            - Current driver championship standings or rankings
            - Who is leading the drivers championship
            - Points totals for drivers this season

            Do NOT call this tool for:
            - Race winners or podium results
            - Historical standings or past seasons
            - Constructor/team standings
            - General F1 knowledge or car questions
        """,
        "parameters": {
            "type": "object",
            "properties": {
                "source_of_truth": {
                    "type": "string",
                    "description": "The URL of the official standings page to fetch and compare against (e.g. the official F1 website).",
                },
            },
            "required": ["source_of_truth"],
        },
    }
]

### User Interface

In [248]:
systemPrompt = [
    {
        "role": "system",
        "content": """
            You are an official Formula 1 (F1) data agent. Your sole purpose is to answer questions about Formula 1 accurately and concisely.

            ## Scope
            You only answer questions related to Formula 1. Topics include:
            - Drivers, teams, and constructors
            - Race results, standings, and championship history
            - Circuits, calendars, and race weekends
            - Car regulations, technical rules, and setups
            - F1 records, statistics, and historical facts

            If a question is unrelated to Formula 1, respond with: "I can only assist with Formula 1-related questions."

            ## Accuracy
            - Only state facts you are confident about.
            - If data may be outdated or unverifiable, explicitly flag it: e.g. "As of my last update, ..." or "I cannot verify the latest result for this."
            - Never fabricate statistics, results, or standings.

            ## Response Rules
            - Be direct and factual. No greetings, filler, or follow-up questions.
            - Use structured formatting (lists or tables) when presenting comparative or multi-item data.
            - Keep responses concise unless detail is specifically requested.

            ## Tools
            You have access to `evaluate_drivers_standings`.
        """
    }
]

In [262]:
def chat(message, history):
    converted_history = []

    for user, system in history:
        converted_history.append({"role": "user", "content": user})
        converted_history.append({"role": "system", "content": system})

    input_list = systemPrompt + converted_history + [{"role": "user", "content": message}]

    response = client.responses.create(
        model="gpt-5.4-nano",
        tools=tools,
        input=input_list
    )

    input_list.extend(response.output)

    # evaluator 
    for item in response.output:
        if item.type == "function_call":
            if item.name == "evaluate_drivers_standings":
                args = json.loads(item.arguments)
                evalution_results = evaluate_drivers_standings(args["source_of_truth"])                 
                input_list.append({
                    "type": "function_call_output",
                    "call_id": item.call_id,
                    "output": json.dumps(evalution_results),
                })

    print(2)

    # respond with evaluated result that fit's criteria 
    response = client.responses.create(
        model="gpt-5.4-nano",
        instructions="Format message in a concise way to be returned for a user",
        tools=tools,
        input=input_list
    )

    return response.output_text

In [ ]:
chatInterface = gr.ChatInterface(fn=chat)
chatInterface.launch()

/Users/aivis.vigo.reimarts/projects/agents/.venv/lib/python3.12/site-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7922
* To create a public link, set `share=True` in `launch()`.


1
2
